# Prepare for Deployment
## Personal Cognitive Load & Focus Monitoring System

This notebook prepares the model and artifacts for deployment.

### Objectives:
1. Load the best model from MLflow
2. Export model artifacts
3. Create model metadata
4. Prepare for containerization
5. Validate model can be loaded in production format


In [1]:
# Import libraries
import mlflow
import mlflow.sklearn
from pathlib import Path
import pickle
import json
import sys

# Add project root and src to path
PROJECT_ROOT = Path().resolve().parent
for path in [PROJECT_ROOT, PROJECT_ROOT / "src"]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

# Set paths
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

# Initialize MLflow
MLRUNS_DIR = PROJECT_ROOT / "mlruns"
MLRUNS_DIR.mkdir(exist_ok=True)
mlflow.set_tracking_uri(f"file://{MLRUNS_DIR}")

print(f"Models directory: {MODELS_DIR}")
print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")


Models directory: /home/pnmynh/Documents/personal-cognitive-load-monitor/models
MLflow tracking URI: file:///home/pnmynh/Documents/personal-cognitive-load-monitor/mlruns


In [2]:
# List available models in MLflow
experiment = mlflow.get_experiment_by_name("cognitive-load-prediction")
if experiment:
    runs = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        order_by=["metrics.accuracy DESC"],
    )
    print("Available model runs:")

    if runs.empty:
        print("No runs found.")
        best_run_id = None
    else:
        display_cols = [
            col for col in ["run_id", "metrics.accuracy", "status"] if col in runs.columns
        ]
        if display_cols:
            print(runs[display_cols].head())
        else:
            print(runs.head())

        best_run_id = runs.iloc[0]["run_id"]
        print(f"\nBest run ID: {best_run_id}")
else:
    print("No experiment found. Please train a model first.")
    best_run_id = None


Available model runs:
                             run_id  metrics.accuracy    status
0  97130239c5974ba3bafc0f4b1014a3d5             0.925  FINISHED
1  d73141ce592f43409007e618e0610a04             0.925  FINISHED
2  8a8edea2def4478e83df419dd5d1f590             0.900  FINISHED

Best run ID: 97130239c5974ba3bafc0f4b1014a3d5


/home/pnmynh/.local/share/mise/installs/python/3.13.5/lib/python3.13/site-packages/mlflow/tracking/_tracking_service/utils.py:178: FutureWarning: The filesystem tracking backend (e.g., './mlruns') will be deprecated in February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://github.com/mlflow/mlflow/issues/18534 for more details and migration guidance. For migrating existing data, https://github.com/mlflow/mlflow-export-import can be used.
  return FileStore(store_uri, store_uri)


In [3]:
# Load model from MLflow (if available)
# For AIDE 1, we'll use the rule-based service as the "model"
# This demonstrates the pattern for when you have an ML model

# Option 1: Export rule-based service (current implementation)
from preprocessing_api.service import calculate_cognitive_load

# Save service metadata
model_metadata = {
    "model_type": "rule_based",
    "version": "1.0.0",
    "description": "Rule-based cognitive load prediction",
    "features": [
        "focus_minutes",
        "distraction_minutes",
        "tasks_due",
        "hours_to_deadline"
    ],
    "output": "cognitive_load_level (LOW, MEDIUM, HIGH)"
}

metadata_file = MODELS_DIR / "model_metadata.json"
with open(metadata_file, "w") as f:
    json.dump(model_metadata, f, indent=2)

print("Model metadata saved:")
print(json.dumps(model_metadata, indent=2))


Model metadata saved:
{
  "model_type": "rule_based",
  "version": "1.0.0",
  "description": "Rule-based cognitive load prediction",
  "features": [
    "focus_minutes",
    "distraction_minutes",
    "tasks_due",
    "hours_to_deadline"
  ],
  "output": "cognitive_load_level (LOW, MEDIUM, HIGH)"
}


In [4]:
# Option 2: If using ML model, load from MLflow
if best_run_id:
    try:
        model_uri = f"runs:/{best_run_id}/model"
        model = mlflow.sklearn.load_model(model_uri)
        
        # Save model locally
        model_file = MODELS_DIR / "model.pkl"
        with open(model_file, "wb") as f:
            pickle.dump(model, f)
        
        print(f"Model saved to {model_file}")
        print(f"Model type: {type(model)}")
    except Exception as e:
        print(f"Could not load ML model: {e}")
        print("Using rule-based service instead")
else:
    print("Using rule-based service (no ML model available)")


Model saved to /home/pnmynh/Documents/personal-cognitive-load-monitor/models/model.pkl
Model type: <class 'sklearn.ensemble._forest.RandomForestClassifier'>


In [5]:
# Validate model can be loaded and used
from preprocessing_api.schemas import PredictionRequest
from preprocessing_api.service import calculate_cognitive_load

# Test prediction
test_request = PredictionRequest(
    focus_minutes=120,
    distraction_minutes=30,
    tasks_due=3,
    hours_to_deadline=24.0
)

prediction = calculate_cognitive_load(test_request)
print("Test prediction:")
print(f"Input: {test_request.model_dump()}")
print(f"Output: {prediction.value}")
print("\n✅ Model is ready for deployment!")


Test prediction:
Input: {'focus_minutes': 120, 'distraction_minutes': 30, 'tasks_due': 3, 'hours_to_deadline': 24.0}
Output: MEDIUM

✅ Model is ready for deployment!


## Deployment Checklist

- [x] Model artifacts exported
- [x] Model metadata created
- [x] Model validation passed
- [ ] Docker image built (next step)
- [ ] CI/CD pipeline configured (next step)
- [ ] Kubernetes deployment ready (next step)

### Next Steps:
1. Create Dockerfile for the API
2. Build Docker image
3. Set up CI/CD pipeline
4. Deploy to Kubernetes with Helm
